# Chapter 8: Integrating a custom Triton operation with PyTorch

This notebook is the complete reader-facing code artifact for Chapter 8 of
*GPU Programming with Triton*. It starts with a small decoder-only
TinyStories workload, isolates attention-score normalization, implements
Softpick in readable PyTorch, replaces that reference with Triton forward
and backward kernels, and registers the kernels as a trainable PyTorch
operation.

The learning path is:

1. Build the decoder workload around one row-normalization call site.
2. Use the readable Softpick implementation as the numerical oracle.
3. Validate standalone Triton forward and backward kernels.
4. Register one autograd-aware operation with PyTorch.
5. Test correctness before measuring performance.
6. Exercise the operation in a real training step.

The decoder is an integration workload, not a reproduction of the
paper-scale Softpick experiments. PyTorch still performs the query-key and
attention-value matrix multiplications; Triton replaces only the row-wise
normalization of materialized attention scores.


## 0. Prepare the Colab runtime

In Colab, select **Runtime > Change runtime type > GPU** before running the
notebook. Run the cells from top to bottom. A CPU runtime can execute the
reference path and CPU fallback, but the Triton kernel tests and GPU
measurements will be skipped.

The next cell installs the Hugging Face dataset loader used only by the
optional TinyStories training run. PyTorch and Triton should come from the
same Colab runtime image so their versions remain compatible.

The PyTorch registration cell creates the `tritonbook::softpick` operator.
If you edit and rerun that cell, restart the runtime and run from the top;
PyTorch intentionally rejects duplicate operator registrations.


In [ ]:
%pip install -q datasets


In [ ]:
from __future__ import annotations

import contextlib
import dataclasses
import hashlib
import importlib.metadata
import itertools
import json
import math
import os
import random
import time
import unittest
from collections.abc import Callable
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.library import triton_op, wrap_triton

try:
    import triton
    import triton.language as tl
except ImportError as error:
    raise RuntimeError(
        "This chapter requires Triton. In Colab, select a GPU runtime and "
        "restart the session so PyTorch and Triton come from one image."
    ) from error

_TRITON_IMPORT_ERROR = None
DEFAULT_SOFTPICK_EPSILON = 1e-6

print(f"PyTorch version    : {torch.__version__}")
print(f"CUDA build         : {torch.version.cuda}")
print(f"Triton version     : {importlib.metadata.version('triton')}")
print(f"CUDA available     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU                : {torch.cuda.get_device_name(0)}")
    print(f"Compute capability : {torch.cuda.get_device_capability(0)}")
else:
    print("GPU cells will be skipped. Enable a GPU runtime for Triton execution.")

if not hasattr(torch.library, "triton_op"):
    raise RuntimeError(
        "This notebook needs a PyTorch build that provides "
        "torch.library.triton_op."
    )


## 1. Build the decoder workload

The byte-level decoder keeps four attention steps explicit: project queries,
keys, and values; form materialized query-key scores; normalize each score
row; and multiply the weights by the values. The
`normalize_attention_scores` function is the only integration call site. Its
default path uses the readable Softpick implementation defined in the next
section.


In [ ]:
# Byte-level tokenizer constants. Byte values occupy ids 0 through 255 and the
# end-of-story marker occupies id 256.
BYTE_VOCABULARY_SIZE = 256
END_OF_STORY_TOKEN_ID = 256
TOKENIZER_VOCABULARY_SIZE = BYTE_VOCABULARY_SIZE + 1
TOKENIZER_VERSION = "byte-utf8-v1"

DATASET_ID = "roneneldan/TinyStories"
DATASET_TEXT_FIELD = "text"


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------


@dataclasses.dataclass
class ModelConfig:
    """Model shape. Defaults are sized for a single NVIDIA T4."""

    vocabulary_size: int = TOKENIZER_VOCABULARY_SIZE
    context_length: int = 256
    model_width: int = 256
    num_heads: int = 4
    num_layers: int = 4
    feedforward_width: int = 1024
    attention_dropout: float = 0.0
    residual_dropout: float = 0.0
    tie_embeddings: bool = True
    attention_normalizer: str = "softpick-reference"
    softpick_epsilon: float = DEFAULT_SOFTPICK_EPSILON

    def __post_init__(self) -> None:
        if self.model_width % self.num_heads != 0:
            raise ValueError(
                f"model width {self.model_width} is not divisible by "
                f"{self.num_heads} heads"
            )
        choices = {
            "softmax",
            "softpick-reference",
            "softpick-triton",
        }
        if self.attention_normalizer not in choices:
            raise ValueError(
                "attention_normalizer must be softmax, softpick-reference, "
                "or softpick-triton"
            )
        if not math.isfinite(self.softpick_epsilon) or self.softpick_epsilon <= 0:
            raise ValueError("softpick_epsilon must be a finite positive number")

    @property
    def head_width(self) -> int:
        return self.model_width // self.num_heads


# ---------------------------------------------------------------------------
# The normalization seam
# ---------------------------------------------------------------------------


def normalize_attention_scores(
    attention_scores: torch.Tensor,
    keep_mask: torch.Tensor,
    normalizer: str = "softpick-reference",
    softpick_epsilon: float = DEFAULT_SOFTPICK_EPSILON,
) -> torch.Tensor:
    """Turn raw attention scores into attention weights.

    `attention_scores` has shape [batch, heads, query, key] and may hold any
    floating dtype. `keep_mask` is a broadcastable Boolean tensor that is True
    for positions a query is allowed to attend to.

    Each implementation returns the score tensor's dtype so that the following
    multiplication with V stays in the model's working precision.

    This is the only attention-normalization site in the program.
    """
    if normalizer == "softpick-reference":
        return softpick_reference(
            attention_scores,
            keep_mask,
            softpick_epsilon,
        )
    if normalizer == "softpick-triton":
        return softpick(
            attention_scores,
            keep_mask,
            softpick_epsilon,
        )
    if normalizer != "softmax":
        raise ValueError(f"unknown attention normalizer: {normalizer}")

    scores_float32 = attention_scores.float()
    masked_scores = scores_float32.masked_fill(~keep_mask, float("-inf"))
    attention_weights = torch.softmax(masked_scores, dim=-1)
    return attention_weights.to(attention_scores.dtype)


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------


class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention written with explicit tensor steps."""

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        assert config.model_width % config.num_heads == 0
        self.num_heads = config.num_heads
        self.head_width = config.head_width
        self.score_scale = 1.0 / math.sqrt(config.head_width)
        self.attention_normalizer = config.attention_normalizer
        self.softpick_epsilon = config.softpick_epsilon

        self.query_projection = nn.Linear(config.model_width, config.model_width)
        self.key_projection = nn.Linear(config.model_width, config.model_width)
        self.value_projection = nn.Linear(config.model_width, config.model_width)
        self.output_projection = nn.Linear(config.model_width, config.model_width)
        self.attention_dropout = nn.Dropout(config.attention_dropout)
        self.residual_dropout = nn.Dropout(config.residual_dropout)

        keep_mask = torch.ones(
            config.context_length,
            config.context_length,
            dtype=torch.bool,
        ).tril()
        self.register_buffer("causal_keep_mask", keep_mask.view(1, 1, *keep_mask.shape))

    def split_heads(self, projected: torch.Tensor) -> torch.Tensor:
        """[batch, sequence, width] to [batch, heads, sequence, head width]."""
        batch, sequence, _ = projected.shape
        reshaped = projected.view(batch, sequence, self.num_heads, self.head_width)
        return reshaped.transpose(1, 2)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        batch, sequence, width = hidden_states.shape
        if sequence > self.causal_keep_mask.shape[-1]:
            raise ValueError(
                f"sequence length {sequence} exceeds the configured context "
                f"length {self.causal_keep_mask.shape[-1]}"
            )

        query = self.split_heads(self.query_projection(hidden_states))
        key = self.split_heads(self.key_projection(hidden_states))
        value = self.split_heads(self.value_projection(hidden_states))

        # [batch, heads, sequence, sequence]
        attention_scores = torch.matmul(query, key.transpose(-2, -1))
        attention_scores = attention_scores * self.score_scale

        keep_mask = self.causal_keep_mask[:, :, :sequence, :sequence]
        attention_weights = normalize_attention_scores(
            attention_scores,
            keep_mask,
            self.attention_normalizer,
            self.softpick_epsilon,
        )
        attention_weights = self.attention_dropout(attention_weights)

        context = torch.matmul(attention_weights, value)
        context = context.transpose(1, 2).reshape(batch, sequence, width)
        return self.residual_dropout(self.output_projection(context))


class FeedForward(nn.Module):
    """Two-layer position-wise network with a GELU activation."""

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.input_projection = nn.Linear(
            config.model_width, config.feedforward_width
        )
        self.activation = nn.GELU()
        self.output_projection = nn.Linear(
            config.feedforward_width, config.model_width
        )
        self.dropout = nn.Dropout(config.residual_dropout)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        hidden_states = self.input_projection(hidden_states)
        hidden_states = self.activation(hidden_states)
        hidden_states = self.output_projection(hidden_states)
        return self.dropout(hidden_states)


class TransformerBlock(nn.Module):
    """Pre-normalization block with residual connections."""

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.attention_norm = nn.LayerNorm(config.model_width)
        self.attention = CausalSelfAttention(config)
        self.feedforward_norm = nn.LayerNorm(config.model_width)
        self.feedforward = FeedForward(config)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        hidden_states = hidden_states + self.attention(
            self.attention_norm(hidden_states)
        )
        hidden_states = hidden_states + self.feedforward(
            self.feedforward_norm(hidden_states)
        )
        return hidden_states


class ByteLevelDecoder(nn.Module):
    """Decoder-only language model over byte tokens."""

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(
            config.vocabulary_size, config.model_width
        )
        self.position_embedding = nn.Embedding(
            config.context_length, config.model_width
        )
        self.embedding_dropout = nn.Dropout(config.residual_dropout)
        self.blocks = nn.ModuleList(
            TransformerBlock(config) for _ in range(config.num_layers)
        )
        self.final_norm = nn.LayerNorm(config.model_width)
        self.output_head = nn.Linear(
            config.model_width, config.vocabulary_size, bias=False
        )
        if config.tie_embeddings:
            self.output_head.weight = self.token_embedding.weight

        self.apply(self.initialize_parameters)

    @staticmethod
    def initialize_parameters(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(
        self,
        input_tokens: torch.Tensor,
        target_tokens: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        batch, sequence = input_tokens.shape
        positions = torch.arange(sequence, device=input_tokens.device)

        hidden_states = self.token_embedding(input_tokens)
        hidden_states = hidden_states + self.position_embedding(positions)
        hidden_states = self.embedding_dropout(hidden_states)

        for block in self.blocks:
            hidden_states = block(hidden_states)

        hidden_states = self.final_norm(hidden_states)
        logits = self.output_head(hidden_states)

        loss = None
        if target_tokens is not None:
            loss = F.cross_entropy(
                logits.float().view(-1, logits.shape[-1]),
                target_tokens.reshape(-1),
            )
        return logits, loss

    def count_parameters(self) -> int:
        seen: dict[int, int] = {}
        for parameter in self.parameters():
            if parameter.requires_grad:
                seen[id(parameter)] = parameter.numel()
        return sum(seen.values())


## 2. Write the readable Softpick oracle

A custom kernel needs an independent specification. The reference below
keeps the equation visible in ordinary PyTorch and lets autograd calculate
its derivative. A Boolean `keep_mask` is part of the operation contract:
disallowed positions contribute to neither the maximum nor the numerator
nor the denominator.

Reduced-precision inputs are promoted to float32 for exponentials and
reductions. The result is cast back to the input dtype so the following
attention-value multiplication stays in the model's working precision.


In [ ]:
def _check_metadata(
    attention_scores: torch.Tensor,
    keep_mask: torch.Tensor,
    epsilon: float,
) -> None:
    """Reject input metadata that does not match the operation contract."""
    if not attention_scores.is_floating_point():
        raise TypeError("attention_scores must use a floating-point dtype")
    if keep_mask.dtype is not torch.bool:
        raise TypeError("keep_mask must use torch.bool")
    if attention_scores.device != keep_mask.device:
        raise ValueError("attention_scores and keep_mask must share a device")
    if not math.isfinite(epsilon) or epsilon <= 0.0:
        raise ValueError("epsilon must be a finite positive number")
    try:
        broadcast_shape = torch.broadcast_shapes(
            attention_scores.shape,
            keep_mask.shape,
        )
    except RuntimeError as error:
        raise ValueError(
            "keep_mask must be broadcastable to attention_scores"
        ) from error
    if broadcast_shape != attention_scores.shape:
        raise ValueError(
            "keep_mask must broadcast to exactly the attention_scores shape"
        )


def validate_softpick_inputs(
    attention_scores: torch.Tensor,
    keep_mask: torch.Tensor,
    epsilon: float = DEFAULT_SOFTPICK_EPSILON,
) -> None:
    """Perform the synchronization-heavy checks used by tests and debugging.

    The hot implementation below checks metadata but does not read tensor
    values back to Python. Call this validator at workload boundaries when you
    need an explicit error for non-finite valid scores or fully masked rows.
    """
    _check_metadata(attention_scores, keep_mask, epsilon)
    expanded_mask = torch.broadcast_to(keep_mask, attention_scores.shape)
    valid_scores = attention_scores.masked_select(expanded_mask)

    if not bool(torch.isfinite(valid_scores).all().item()):
        raise ValueError("all unmasked attention scores must be finite")
    if not bool(expanded_mask.any(dim=-1).all().item()):
        raise ValueError("every Softpick row must contain an unmasked value")


def softpick_reference(
    attention_scores: torch.Tensor,
    keep_mask: torch.Tensor,
    epsilon: float = DEFAULT_SOFTPICK_EPSILON,
) -> torch.Tensor:
    """Normalize the last dimension with masked, numerically stable Softpick.

    Float16 and bfloat16 inputs are promoted to float32 for exponentials and
    reductions. Float64 is retained so that finite-difference and derivative
    tests can request the extra precision. The result has the same shape,
    device, and dtype as ``attention_scores``.

    A fully masked row is outside the workload contract. The arithmetic still
    returns zeros for that row so invalid input cannot create NaNs silently;
    ``validate_softpick_inputs`` provides the stricter debugging check.
    """
    _check_metadata(attention_scores, keep_mask, epsilon)

    expanded_mask = torch.broadcast_to(keep_mask, attention_scores.shape)
    compute_dtype = (
        torch.float64 if attention_scores.dtype is torch.float64 else torch.float32
    )
    scores = attention_scores.to(compute_dtype)

    masked_for_max = torch.where(
        expanded_mask,
        scores,
        torch.full_like(scores, float("-inf")),
    )
    row_has_value = expanded_mask.any(dim=-1, keepdim=True)
    row_maximum = masked_for_max.amax(dim=-1, keepdim=True)
    safe_maximum = torch.where(
        row_has_value,
        row_maximum,
        torch.zeros_like(row_maximum),
    )

    safe_scores = torch.where(expanded_mask, scores, torch.zeros_like(scores))
    shifted = torch.exp(safe_scores - safe_maximum) - torch.exp(-safe_maximum)
    shifted = torch.where(
        expanded_mask,
        shifted,
        torch.zeros_like(shifted),
    )

    numerator = torch.relu(shifted)
    denominator = (
        shifted.abs().sum(
            dim=-1,
            keepdim=True,
        )
        + epsilon
    )
    weights = numerator / denominator
    weights = torch.where(expanded_mask, weights, torch.zeros_like(weights))
    return weights.to(attention_scores.dtype)


In [ ]:
scores = torch.tensor(
    [[[[math.log(2.0), 0.0, -math.log(2.0), 10.0]]]],
    dtype=torch.float64,
)
keep_mask = torch.tensor([True, True, True, False])
weights = softpick_reference(scores, keep_mask)

denominator = 0.75 + DEFAULT_SOFTPICK_EPSILON
expected = torch.tensor(
    [[[[0.5 / denominator, 0.0, 0.0, 0.0]]]],
    dtype=torch.float64,
)
torch.testing.assert_close(weights, expected, rtol=1e-12, atol=1e-12)
print("PASS: worked row =", weights.flatten().tolist())


## 3. Implement standalone Triton forward and backward kernels

One Triton program owns one `[K]` score row. The launch grid flattens the
batch, head, and query dimensions. Each program reconstructs those indices
to address a broadcast mask, pads the key dimension to a power of two, and
excludes masked and padding lanes from every reduction.

The standalone forward launcher deliberately refuses a gradient-tracked
input. A raw Triton launch does not add an autograd node; silently accepting
such an input would break the training graph. The backward launcher is
tested directly against the vector-Jacobian product produced by the
PyTorch oracle.


In [ ]:
MAX_FUSED_ROW_WIDTH = 65_536


if triton is not None:

    @triton.jit
    def _softpick_forward_kernel(
        scores_pointer,
        mask_pointer,
        output_pointer,
        rows_per_batch_head,
        rows_per_head,
        row_width: tl.constexpr,
        mask_stride_batch,
        mask_stride_head,
        mask_stride_query,
        mask_stride_key,
        epsilon: tl.constexpr,
        block_size: tl.constexpr,
    ):
        row = tl.program_id(axis=0)
        batch_index = row // rows_per_batch_head
        within_batch = row % rows_per_batch_head
        head_index = within_batch // rows_per_head
        query_index = within_batch % rows_per_head

        columns = tl.arange(0, block_size)
        in_bounds = columns < row_width
        mask_offsets = (
            batch_index * mask_stride_batch
            + head_index * mask_stride_head
            + query_index * mask_stride_query
            + columns * mask_stride_key
        )
        allowed = (
            tl.load(
                mask_pointer + mask_offsets,
                mask=in_bounds,
                other=0,
            )
            != 0
        )
        active = in_bounds & allowed

        score_offsets = row * row_width + columns
        scores = tl.load(
            scores_pointer + score_offsets,
            mask=active,
            other=-float("inf"),
        ).to(tl.float32)

        row_has_value = tl.sum(active.to(tl.int32), axis=0) > 0
        row_maximum = tl.max(scores, axis=0)
        safe_maximum = tl.where(row_has_value, row_maximum, 0.0)

        shifted = tl.exp(scores - safe_maximum) - tl.exp(-safe_maximum)
        shifted = tl.where(active, shifted, 0.0)
        numerator = tl.maximum(shifted, 0.0)
        denominator = tl.sum(tl.abs(shifted), axis=0) + epsilon
        weights = tl.where(active, numerator / denominator, 0.0)

        tl.store(output_pointer + score_offsets, weights, mask=in_bounds)


    @triton.jit
    def _softpick_backward_kernel(
        scores_pointer,
        mask_pointer,
        grad_output_pointer,
        grad_scores_pointer,
        rows_per_batch_head,
        rows_per_head,
        row_width: tl.constexpr,
        mask_stride_batch,
        mask_stride_head,
        mask_stride_query,
        mask_stride_key,
        epsilon: tl.constexpr,
        block_size: tl.constexpr,
    ):
        row = tl.program_id(axis=0)
        batch_index = row // rows_per_batch_head
        within_batch = row % rows_per_batch_head
        head_index = within_batch // rows_per_head
        query_index = within_batch % rows_per_head

        columns = tl.arange(0, block_size)
        in_bounds = columns < row_width
        mask_offsets = (
            batch_index * mask_stride_batch
            + head_index * mask_stride_head
            + query_index * mask_stride_query
            + columns * mask_stride_key
        )
        allowed = (
            tl.load(
                mask_pointer + mask_offsets,
                mask=in_bounds,
                other=0,
            )
            != 0
        )
        active = in_bounds & allowed

        score_offsets = row * row_width + columns
        scores = tl.load(
            scores_pointer + score_offsets,
            mask=active,
            other=-float("inf"),
        ).to(tl.float32)
        grad_output = tl.load(
            grad_output_pointer + score_offsets,
            mask=active,
            other=0.0,
        ).to(tl.float32)

        row_has_value = tl.sum(active.to(tl.int32), axis=0) > 0
        row_maximum = tl.max(scores, axis=0)
        safe_maximum = tl.where(row_has_value, row_maximum, 0.0)

        exponential = tl.exp(scores - safe_maximum)
        shifted = exponential - tl.exp(-safe_maximum)
        shifted = tl.where(active, shifted, 0.0)
        numerator = tl.maximum(shifted, 0.0)
        denominator = tl.sum(tl.abs(shifted), axis=0) + epsilon
        weights = tl.where(active, numerator / denominator, 0.0)

        weighted_upstream = tl.sum(grad_output * weights, axis=0)
        positive_upstream = tl.where(scores > 0.0, grad_output, 0.0)
        score_sign = tl.where(
            scores > 0.0,
            1.0,
            tl.where(scores < 0.0, -1.0, 0.0),
        )
        core_gradient = exponential / denominator * (
            positive_upstream - score_sign * weighted_upstream
        )

        is_maximum = active & (scores == safe_maximum)
        maximum_count = tl.sum(is_maximum.to(tl.float32), axis=0)
        safe_maximum_count = tl.maximum(maximum_count, 1.0)
        maximum_share = (
            is_maximum.to(tl.float32) / safe_maximum_count
        )
        maximum_correction = (
            maximum_share
            * epsilon
            / denominator
            * weighted_upstream
        )
        grad_scores = tl.where(
            active,
            core_gradient - maximum_correction,
            0.0,
        )

        tl.store(
            grad_scores_pointer + score_offsets,
            grad_scores,
            mask=in_bounds,
        )


def triton_is_available() -> bool:
    """Return whether the Python package needed to launch the kernel exists."""
    return _TRITON_IMPORT_ERROR is None


def _check_inputs(
    attention_scores: torch.Tensor,
    keep_mask: torch.Tensor,
    epsilon: float,
) -> torch.Tensor:
    if _TRITON_IMPORT_ERROR is not None:
        raise RuntimeError(
            "Triton is not installed in this Python environment"
        ) from _TRITON_IMPORT_ERROR
    if not attention_scores.is_cuda:
        raise ValueError("the Triton Softpick kernel requires CUDA scores")
    if attention_scores.ndim != 4:
        raise ValueError("attention_scores must have shape [B, H, Q, K]")
    if not attention_scores.is_contiguous():
        raise ValueError("attention_scores must be contiguous")
    if attention_scores.dtype not in {
        torch.float16,
        torch.bfloat16,
        torch.float32,
    }:
        raise TypeError("scores must use float16, bfloat16, or float32")
    if keep_mask.dtype is not torch.bool:
        raise TypeError("keep_mask must use torch.bool")
    if keep_mask.device != attention_scores.device:
        raise ValueError("attention_scores and keep_mask must share a device")
    if not math.isfinite(epsilon) or epsilon <= 0.0:
        raise ValueError("epsilon must be a finite positive number")
    if attention_scores.shape[-1] == 0:
        raise ValueError("the key dimension must contain at least one value")
    try:
        expanded_mask = torch.broadcast_to(keep_mask, attention_scores.shape)
    except RuntimeError as error:
        raise ValueError(
            "keep_mask must be broadcastable to attention_scores"
        ) from error

    block_size = triton.next_power_of_2(attention_scores.shape[-1])
    if block_size > MAX_FUSED_ROW_WIDTH:
        raise ValueError(
            f"key width {attention_scores.shape[-1]} exceeds the teaching "
            f"kernel limit of {MAX_FUSED_ROW_WIDTH}"
        )
    return expanded_mask


def softpick_triton_forward(
    attention_scores: torch.Tensor,
    keep_mask: torch.Tensor,
    epsilon: float = DEFAULT_SOFTPICK_EPSILON,
) -> torch.Tensor:
    """Run the forward-only Triton kernel over contiguous ``[B, H, Q, K]``.

    The explicit gradient guard prevents this standalone launcher from
    silently breaking a training graph. Use the registered
    ``softpick`` operation when autograd is required.
    """
    expanded_mask = _check_inputs(attention_scores, keep_mask, epsilon)
    if torch.is_grad_enabled() and attention_scores.requires_grad:
        raise RuntimeError(
            "The standalone Triton launcher is forward-only. Use "
            "softpick(...) for autograd."
        )

    batch, heads, query_rows, row_width = attention_scores.shape
    output = torch.empty_like(attention_scores)
    block_size = triton.next_power_of_2(row_width)
    grid = (batch * heads * query_rows,)

    _softpick_forward_kernel[grid](
        attention_scores,
        expanded_mask,
        output,
        heads * query_rows,
        query_rows,
        row_width=row_width,
        mask_stride_batch=expanded_mask.stride(0),
        mask_stride_head=expanded_mask.stride(1),
        mask_stride_query=expanded_mask.stride(2),
        mask_stride_key=expanded_mask.stride(3),
        epsilon=epsilon,
        block_size=block_size,
        num_warps=4,
    )
    return output


def softpick_triton_backward(
    attention_scores: torch.Tensor,
    keep_mask: torch.Tensor,
    grad_output: torch.Tensor,
    epsilon: float = DEFAULT_SOFTPICK_EPSILON,
) -> torch.Tensor:
    """Apply the Softpick vector-Jacobian product to one upstream tensor.

    This launcher computes the gradient with respect to ``attention_scores``.
    It does not attach either Triton kernel to PyTorch autograd; registration
    is a separate integration step.
    """
    expanded_mask = _check_inputs(attention_scores, keep_mask, epsilon)
    if grad_output.shape != attention_scores.shape:
        raise ValueError("grad_output must match attention_scores.shape")
    if grad_output.device != attention_scores.device:
        raise ValueError("grad_output and attention_scores must share a device")
    if grad_output.dtype != attention_scores.dtype:
        raise TypeError("grad_output and attention_scores must share a dtype")

    batch, heads, query_rows, row_width = attention_scores.shape
    contiguous_grad_output = grad_output.contiguous()
    grad_scores = torch.empty_like(attention_scores)
    block_size = triton.next_power_of_2(row_width)
    grid = (batch * heads * query_rows,)

    _softpick_backward_kernel[grid](
        attention_scores,
        expanded_mask,
        contiguous_grad_output,
        grad_scores,
        heads * query_rows,
        query_rows,
        row_width=row_width,
        mask_stride_batch=expanded_mask.stride(0),
        mask_stride_head=expanded_mask.stride(1),
        mask_stride_query=expanded_mask.stride(2),
        mask_stride_key=expanded_mask.stride(3),
        epsilon=epsilon,
        block_size=block_size,
        num_warps=4,
    )
    return grad_scores


In [ ]:
if torch.cuda.is_available():
    torch.manual_seed(1234)
    kernel_scores = torch.randn(
        2, 3, 5, 37, device="cuda", dtype=torch.float32, requires_grad=True
    )
    query = torch.arange(5, device="cuda").view(5, 1)
    key = torch.arange(37, device="cuda").view(1, 37)
    kernel_mask = (key <= query).view(1, 1, 5, 37)
    upstream = torch.randn_like(kernel_scores)

    expected_weights = softpick_reference(kernel_scores, kernel_mask)
    expected_gradient = torch.autograd.grad(
        expected_weights, kernel_scores, upstream
    )[0]
    actual_weights = softpick_triton_forward(
        kernel_scores.detach(), kernel_mask
    )
    actual_gradient = softpick_triton_backward(
        kernel_scores.detach(), kernel_mask, upstream
    )

    torch.testing.assert_close(
        actual_weights, expected_weights.detach(), rtol=2e-4, atol=2e-5
    )
    torch.testing.assert_close(
        actual_gradient, expected_gradient, rtol=3e-4, atol=3e-5
    )
    print("PASS: standalone forward and backward match at K=37")
else:
    print("SKIP: standalone Triton check requires a GPU runtime")


## 4. Register one trainable PyTorch operation

The forward and backward kernels now sit behind named PyTorch operations.
The forward operation records the original scores and expanded mask. When
autograd supplies an upstream gradient, its formula calls the registered
backward operation and returns the score gradient to the query-key matrix
multiplication.

CPU kernels use the readable PyTorch equations under the same public
`softpick` function. This fallback keeps the model and smoke tests useful
without CUDA while the CUDA dispatch remains the handwritten Triton path.
The operation supports first-order training gradients; the chapter does not
register a derivative for the backward operation itself.


In [ ]:
def _check_operation_inputs(
    attention_scores: torch.Tensor,
    keep_mask: torch.Tensor,
    epsilon: float,
) -> torch.Tensor:
    """Validate the shared CPU and CUDA operation contract."""
    _check_metadata(attention_scores, keep_mask, epsilon)
    if attention_scores.ndim != 4:
        raise ValueError("attention_scores must have shape [B, H, Q, K]")
    if not attention_scores.is_contiguous():
        raise ValueError("attention_scores must be contiguous")
    if attention_scores.dtype not in {
        torch.float16,
        torch.bfloat16,
        torch.float32,
    }:
        raise TypeError("scores must use float16, bfloat16, or float32")
    if attention_scores.device.type not in {"cpu", "cuda"}:
        raise ValueError("Softpick supports CPU and CUDA tensors")
    if attention_scores.shape[-1] == 0:
        raise ValueError("the key dimension must contain at least one value")

    row_width = attention_scores.shape[-1]
    if row_width > MAX_FUSED_ROW_WIDTH:
        raise ValueError(
            f"key width {row_width} exceeds the teaching kernel limit of "
            f"{MAX_FUSED_ROW_WIDTH}"
        )
    if attention_scores.is_cuda and not triton_is_available():
        raise RuntimeError("Triton is required for CUDA Softpick")
    return torch.broadcast_to(keep_mask, attention_scores.shape)


def _softpick_backward_reference(
    attention_scores: torch.Tensor,
    expanded_mask: torch.Tensor,
    grad_output: torch.Tensor,
    epsilon: float,
) -> torch.Tensor:
    """Apply the Section 8.7 vector-Jacobian product with PyTorch ops."""
    compute_dtype = (
        torch.float64
        if attention_scores.dtype is torch.float64
        else torch.float32
    )
    scores = attention_scores.to(compute_dtype)
    safe_scores = torch.where(
        expanded_mask,
        scores,
        torch.zeros_like(scores),
    )
    upstream = torch.where(
        expanded_mask,
        grad_output.to(compute_dtype),
        torch.zeros_like(scores),
    )
    masked_scores = torch.where(
        expanded_mask,
        scores,
        torch.full_like(scores, float("-inf")),
    )
    row_has_value = expanded_mask.any(dim=-1, keepdim=True)
    row_maximum = masked_scores.amax(dim=-1, keepdim=True)
    safe_maximum = torch.where(
        row_has_value,
        row_maximum,
        torch.zeros_like(row_maximum),
    )

    exponential = torch.exp(masked_scores - safe_maximum)
    shifted = exponential - torch.exp(-safe_maximum)
    shifted = torch.where(
        expanded_mask,
        shifted,
        torch.zeros_like(shifted),
    )
    denominator = shifted.abs().sum(dim=-1, keepdim=True) + epsilon
    weights = torch.relu(shifted) / denominator
    weights = torch.where(
        expanded_mask,
        weights,
        torch.zeros_like(weights),
    )

    weighted_upstream = (upstream * weights).sum(dim=-1, keepdim=True)
    positive_upstream = torch.where(
        safe_scores > 0.0,
        upstream,
        torch.zeros_like(upstream),
    )
    score_sign = torch.sign(safe_scores)
    core_gradient = exponential / denominator * (
        positive_upstream - score_sign * weighted_upstream
    )

    is_maximum = expanded_mask & (safe_scores == safe_maximum)
    maximum_count = is_maximum.sum(dim=-1, keepdim=True).to(compute_dtype)
    safe_maximum_count = maximum_count.clamp_min(1.0)
    maximum_share = is_maximum.to(compute_dtype) / safe_maximum_count
    maximum_correction = (
        maximum_share
        * epsilon
        / denominator
        * weighted_upstream
    )
    grad_scores = torch.where(
        expanded_mask,
        core_gradient - maximum_correction,
        torch.zeros_like(core_gradient),
    )
    return grad_scores.to(attention_scores.dtype)


@triton_op("tritonbook::softpick_backward", mutates_args={})
def _softpick_backward_registered(
    attention_scores: torch.Tensor,
    expanded_mask: torch.Tensor,
    grad_output: torch.Tensor,
    epsilon: float,
) -> torch.Tensor:
    batch, heads, query_rows, row_width = attention_scores.shape
    contiguous_grad_output = grad_output.contiguous()
    grad_scores = torch.empty_like(attention_scores)
    block_size = triton.next_power_of_2(row_width)
    grid = (batch * heads * query_rows,)

    backward_kernel = wrap_triton(
        _softpick_backward_kernel
    )
    backward_kernel[grid](
        attention_scores,
        expanded_mask,
        contiguous_grad_output,
        grad_scores,
        heads * query_rows,
        query_rows,
        row_width=row_width,
        mask_stride_batch=expanded_mask.stride(0),
        mask_stride_head=expanded_mask.stride(1),
        mask_stride_query=expanded_mask.stride(2),
        mask_stride_key=expanded_mask.stride(3),
        epsilon=epsilon,
        block_size=block_size,
        num_warps=4,
    )
    return grad_scores


@_softpick_backward_registered.register_kernel("cpu")
def _softpick_backward_cpu(
    attention_scores: torch.Tensor,
    expanded_mask: torch.Tensor,
    grad_output: torch.Tensor,
    epsilon: float,
) -> torch.Tensor:
    return _softpick_backward_reference(
        attention_scores,
        expanded_mask,
        grad_output,
        epsilon,
    )


@triton_op("tritonbook::softpick", mutates_args={})
def _softpick_registered(
    attention_scores: torch.Tensor,
    expanded_mask: torch.Tensor,
    epsilon: float,
) -> torch.Tensor:
    batch, heads, query_rows, row_width = attention_scores.shape
    output = torch.empty_like(attention_scores)
    block_size = triton.next_power_of_2(row_width)
    grid = (batch * heads * query_rows,)

    forward_kernel = wrap_triton(
        _softpick_forward_kernel
    )
    forward_kernel[grid](
        attention_scores,
        expanded_mask,
        output,
        heads * query_rows,
        query_rows,
        row_width=row_width,
        mask_stride_batch=expanded_mask.stride(0),
        mask_stride_head=expanded_mask.stride(1),
        mask_stride_query=expanded_mask.stride(2),
        mask_stride_key=expanded_mask.stride(3),
        epsilon=epsilon,
        block_size=block_size,
        num_warps=4,
    )
    return output


@_softpick_registered.register_kernel("cpu")
def _softpick_cpu(
    attention_scores: torch.Tensor,
    expanded_mask: torch.Tensor,
    epsilon: float,
) -> torch.Tensor:
    return softpick_reference(
        attention_scores,
        expanded_mask,
        epsilon,
    )


def _setup_softpick_context(ctx, inputs, output) -> None:
    attention_scores, expanded_mask, epsilon = inputs
    ctx.save_for_backward(attention_scores, expanded_mask)
    ctx.epsilon = epsilon


def _softpick_autograd_backward(ctx, grad_output):
    attention_scores, expanded_mask = ctx.saved_tensors
    grad_scores = _softpick_backward_registered(
        attention_scores,
        expanded_mask,
        grad_output,
        ctx.epsilon,
    )
    return grad_scores, None, None


_softpick_registered.register_autograd(
    _softpick_autograd_backward,
    setup_context=_setup_softpick_context,
)


def softpick(
    attention_scores: torch.Tensor,
    keep_mask: torch.Tensor,
    epsilon: float = DEFAULT_SOFTPICK_EPSILON,
) -> torch.Tensor:
    """Run the registered Softpick operation on CPU or CUDA tensors."""
    expanded_mask = _check_operation_inputs(
        attention_scores,
        keep_mask,
        epsilon,
    )
    return _softpick_registered(
        attention_scores,
        expanded_mask,
        epsilon,
    )


In [ ]:
torch.manual_seed(1234)
registered_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
registered_values = torch.randn(
    2, 3, 5, 37, device=registered_device, dtype=torch.float32
)
registered_mask = torch.ones(
    1, 1, 5, 37, device=registered_device, dtype=torch.bool
).tril()
registered_upstream = torch.randn_like(registered_values)

reference_scores = registered_values.clone().requires_grad_()
operation_scores = registered_values.clone().requires_grad_()
expected_weights = softpick_reference(reference_scores, registered_mask)
expected_gradient = torch.autograd.grad(
    expected_weights, reference_scores, registered_upstream
)[0]
actual_weights = softpick(operation_scores, registered_mask)
actual_gradient = torch.autograd.grad(
    actual_weights, operation_scores, registered_upstream
)[0]

tolerance = 3e-4 if registered_device.type == "cuda" else 3e-4
torch.testing.assert_close(
    actual_weights, expected_weights, rtol=tolerance, atol=3e-5
)
torch.testing.assert_close(
    actual_gradient, expected_gradient, rtol=tolerance, atol=3e-5
)
print(f"PASS: registered forward and backward match on {registered_device}")


## 5. Run the complete correctness suite

The suite checks the equation, masking, broadcasting, dtype preservation,
invalid rows, CPU dispatch, awkward row widths, reduced precision,
standalone backward, tied maxima, registered autograd, `opcheck`, and
`torch.compile`. GPU-only cases report as skipped on a CPU runtime. A
benchmark is meaningful only after this suite passes on the same software
and hardware environment.


In [ ]:
class SoftpickReferenceTests(unittest.TestCase):
    def test_worked_row_matches_the_equation(self) -> None:
        scores = torch.tensor(
            [[[[math.log(2.0), 0.0, -math.log(2.0), 10.0]]]],
            dtype=torch.float64,
        )
        keep_mask = torch.tensor([True, True, True, False])

        actual = softpick_reference(scores, keep_mask)
        denominator = 0.75 + DEFAULT_SOFTPICK_EPSILON
        expected = torch.tensor(
            [[[[0.5 / denominator, 0.0, 0.0, 0.0]]]],
            dtype=torch.float64,
        )

        torch.testing.assert_close(actual, expected, rtol=1e-12, atol=1e-12)

    def test_masked_large_value_changes_nothing(self) -> None:
        visible_scores = torch.tensor([[[[1.0, 2.0]]]])
        scores = torch.tensor([[[[1.0, 2.0, 10_000.0]]]])
        visible_mask = torch.ones_like(visible_scores, dtype=torch.bool)
        keep_mask = torch.tensor([True, True, False])

        expected = softpick_reference(visible_scores, visible_mask)
        actual = softpick_reference(scores, keep_mask)

        torch.testing.assert_close(actual[..., :2], expected)
        self.assertEqual(actual[..., 2].item(), 0.0)

    def test_zero_and_negative_rows_produce_zero_weights(self) -> None:
        scores = torch.tensor(
            [[[[0.0, 0.0, 0.0], [-1.0, -2.0, -3.0]]]],
        )
        keep_mask = torch.ones_like(scores, dtype=torch.bool)

        actual = softpick_reference(scores, keep_mask)

        torch.testing.assert_close(actual, torch.zeros_like(actual))

    def test_broadcast_mask_and_dtype_contract(self) -> None:
        scores = torch.randn(2, 3, 5, 5, dtype=torch.float16)
        keep_mask = torch.ones(1, 1, 5, 5, dtype=torch.bool).tril()

        actual = softpick_reference(scores, keep_mask)

        self.assertEqual(actual.shape, scores.shape)
        self.assertEqual(actual.dtype, scores.dtype)
        self.assertEqual(actual.device, scores.device)
        disallowed = torch.broadcast_to(~keep_mask, scores.shape)
        self.assertTrue(bool((actual.masked_select(disallowed) == 0).all()))

    def test_reference_retains_autograd_path(self) -> None:
        scores = torch.tensor(
            [[[[2.0, 1.0, 0.0]]]],
            dtype=torch.float64,
            requires_grad=True,
        )
        keep_mask = torch.ones_like(scores, dtype=torch.bool)

        weights = softpick_reference(scores, keep_mask)
        loss = (weights * torch.tensor([1.0, 2.0, 3.0])).sum()
        loss.backward()

        self.assertIsNotNone(scores.grad)
        self.assertTrue(bool(torch.isfinite(scores.grad).all()))

    def test_registered_cpu_fallback_retains_autograd_path(self) -> None:
        torch.manual_seed(1234)
        values = torch.randn(2, 3, 4, 4, dtype=torch.float32)
        keep_mask = torch.ones(1, 1, 4, 4, dtype=torch.bool).tril()
        grad_output = torch.randn_like(values)
        reference_scores = values.clone().requires_grad_()
        registered_scores = values.clone().requires_grad_()

        expected_weights = softpick_reference(reference_scores, keep_mask)
        expected_gradient = torch.autograd.grad(
            expected_weights,
            reference_scores,
            grad_output,
        )[0]
        actual_weights = softpick(registered_scores, keep_mask)
        actual_gradient = torch.autograd.grad(
            actual_weights,
            registered_scores,
            grad_output,
        )[0]

        torch.testing.assert_close(actual_weights, expected_weights)
        torch.testing.assert_close(
            actual_gradient,
            expected_gradient,
            rtol=3e-4,
            atol=3e-5,
        )

    def test_defensive_all_masked_result_is_zero(self) -> None:
        scores = torch.randn(2, 4)
        keep_mask = torch.zeros_like(scores, dtype=torch.bool)

        actual = softpick_reference(scores, keep_mask)

        torch.testing.assert_close(actual, torch.zeros_like(actual))

    def test_validator_rejects_all_masked_row(self) -> None:
        scores = torch.randn(2, 4)
        keep_mask = torch.tensor(
            [[True, False, False, False], [False, False, False, False]]
        )

        with self.assertRaisesRegex(ValueError, "every Softpick row"):
            validate_softpick_inputs(scores, keep_mask)

    def test_validator_ignores_nonfinite_masked_score(self) -> None:
        scores = torch.tensor([[1.0, float("nan")]])
        keep_mask = torch.tensor([[True, False]])

        validate_softpick_inputs(scores, keep_mask)

    def test_validator_rejects_nonfinite_valid_score(self) -> None:
        scores = torch.tensor([[1.0, float("nan")]])
        keep_mask = torch.tensor([[True, True]])

        with self.assertRaisesRegex(ValueError, "must be finite"):
            validate_softpick_inputs(scores, keep_mask)


@unittest.skipUnless(
    torch.cuda.is_available() and triton_is_available(),
    "CUDA and Triton are required for kernel tests",
)
class SoftpickTritonTests(unittest.TestCase):
    def test_awkward_row_width_matches_reference(self) -> None:
        torch.manual_seed(1234)
        scores = torch.randn(
            3,
            2,
            5,
            37,
            device="cuda",
            dtype=torch.float32,
        )
        query = torch.arange(5, device="cuda").view(5, 1)
        key = torch.arange(37, device="cuda").view(1, 37)
        keep_mask = (key <= query).view(1, 1, 5, 37)

        expected = softpick_reference(scores, keep_mask)
        actual = softpick_triton_forward(scores, keep_mask)

        torch.testing.assert_close(actual, expected, rtol=2e-4, atol=2e-5)

    def test_float16_output_matches_reference(self) -> None:
        torch.manual_seed(1234)
        scores = torch.randn(
            2,
            4,
            64,
            64,
            device="cuda",
            dtype=torch.float16,
        )
        keep_mask = torch.ones(
            1,
            1,
            64,
            64,
            device="cuda",
            dtype=torch.bool,
        ).tril()

        expected = softpick_reference(scores, keep_mask)
        actual = softpick_triton_forward(scores, keep_mask)

        torch.testing.assert_close(actual, expected, rtol=2e-3, atol=2e-3)

    def test_forward_only_wrapper_rejects_gradient_use(self) -> None:
        scores = torch.randn(
            1,
            1,
            8,
            8,
            device="cuda",
            requires_grad=True,
        )
        keep_mask = torch.ones_like(scores, dtype=torch.bool).tril()

        with self.assertRaisesRegex(RuntimeError, "forward-only"):
            softpick_triton_forward(scores, keep_mask)

    def test_backward_matches_reference_at_awkward_width(self) -> None:
        torch.manual_seed(1234)
        scores = torch.randn(
            2,
            3,
            5,
            37,
            device="cuda",
            dtype=torch.float32,
            requires_grad=True,
        )
        query = torch.arange(5, device="cuda").view(5, 1)
        key = torch.arange(37, device="cuda").view(1, 37)
        keep_mask = (key <= query).view(1, 1, 5, 37)
        grad_output = torch.randn_like(scores)

        weights = softpick_reference(scores, keep_mask)
        expected = torch.autograd.grad(
            weights,
            scores,
            grad_output,
        )[0]
        actual = softpick_triton_backward(
            scores.detach(),
            keep_mask,
            grad_output,
        )

        torch.testing.assert_close(actual, expected, rtol=3e-4, atol=3e-5)

    def test_backward_matches_reference_in_float16(self) -> None:
        torch.manual_seed(1234)
        scores = torch.randn(
            2,
            4,
            32,
            32,
            device="cuda",
            dtype=torch.float16,
            requires_grad=True,
        )
        keep_mask = torch.ones(
            1,
            1,
            32,
            32,
            device="cuda",
            dtype=torch.bool,
        ).tril()
        grad_output = torch.randn_like(scores).transpose(-1, -2)

        weights = softpick_reference(scores, keep_mask)
        expected = torch.autograd.grad(
            weights,
            scores,
            grad_output,
        )[0]
        actual = softpick_triton_backward(
            scores.detach(),
            keep_mask,
            grad_output,
        )

        torch.testing.assert_close(actual, expected, rtol=4e-3, atol=4e-3)

    def test_backward_keeps_masked_gradients_at_zero(self) -> None:
        scores = torch.tensor(
            [[[[1.5, -0.5, 10_000.0]]]],
            device="cuda",
            requires_grad=True,
        )
        keep_mask = torch.tensor(
            [True, True, False],
            device="cuda",
        )
        grad_output = torch.tensor(
            [[[[0.25, -1.0, 100.0]]]],
            device="cuda",
        )

        actual = softpick_triton_backward(
            scores.detach(),
            keep_mask,
            grad_output,
        )

        self.assertEqual(actual[..., 2].item(), 0.0)
        self.assertNotEqual(actual[..., 1].item(), 0.0)

    def test_backward_includes_the_epsilon_maximum_correction(self) -> None:
        scores = torch.tensor(
            [[[[1.2, -0.7, 0.3]]]],
            device="cuda",
            dtype=torch.float32,
            requires_grad=True,
        )
        keep_mask = torch.ones_like(scores, dtype=torch.bool)
        grad_output = torch.tensor(
            [[[[0.5, -1.0, 2.0]]]],
            device="cuda",
        )
        epsilon = 0.25

        weights = softpick_reference(scores, keep_mask, epsilon)
        expected = torch.autograd.grad(
            weights,
            scores,
            grad_output,
        )[0]
        actual = softpick_triton_backward(
            scores.detach(),
            keep_mask,
            grad_output,
            epsilon,
        )

        torch.testing.assert_close(actual, expected, rtol=3e-4, atol=3e-5)

    def test_backward_splits_a_tied_maximum_like_amax(self) -> None:
        scores = torch.tensor(
            [[[[1.25, 1.25, -0.75]]]],
            device="cuda",
            dtype=torch.float32,
            requires_grad=True,
        )
        keep_mask = torch.ones_like(scores, dtype=torch.bool)
        grad_output = torch.tensor(
            [[[[0.5, -1.0, 2.0]]]],
            device="cuda",
        )
        epsilon = 0.25

        weights = softpick_reference(scores, keep_mask, epsilon)
        expected = torch.autograd.grad(
            weights,
            scores,
            grad_output,
        )[0]
        actual = softpick_triton_backward(
            scores.detach(),
            keep_mask,
            grad_output,
            epsilon,
        )

        torch.testing.assert_close(actual, expected, rtol=3e-4, atol=3e-5)

    def test_registered_operation_connects_forward_and_backward(self) -> None:
        torch.manual_seed(1234)
        values = torch.randn(
            2,
            3,
            5,
            37,
            device="cuda",
            dtype=torch.float16,
        )
        query = torch.arange(5, device="cuda").view(5, 1)
        key = torch.arange(37, device="cuda").view(1, 37)
        keep_mask = (key <= query).view(1, 1, 5, 37)
        grad_output = torch.randn_like(values)
        reference_scores = values.clone().requires_grad_()
        registered_scores = values.clone().requires_grad_()

        expected_weights = softpick_reference(reference_scores, keep_mask)
        expected_gradient = torch.autograd.grad(
            expected_weights,
            reference_scores,
            grad_output,
        )[0]
        actual_weights = softpick(registered_scores, keep_mask)
        actual_gradient = torch.autograd.grad(
            actual_weights,
            registered_scores,
            grad_output,
        )[0]

        torch.testing.assert_close(
            actual_weights,
            expected_weights,
            rtol=2e-3,
            atol=2e-3,
        )
        torch.testing.assert_close(
            actual_gradient,
            expected_gradient,
            rtol=4e-3,
            atol=4e-3,
        )

    def test_registered_operation_passes_opcheck(self) -> None:
        scores = torch.randn(
            1,
            1,
            4,
            4,
            device="cuda",
            requires_grad=True,
        )
        keep_mask = torch.ones_like(scores, dtype=torch.bool).tril()

        torch.library.opcheck(
            torch.ops.tritonbook.softpick.default,
            (
                scores,
                keep_mask,
                DEFAULT_SOFTPICK_EPSILON,
            ),
            test_utils=(
                "test_schema",
                "test_autograd_registration",
                "test_faketensor",
            ),
        )

    def test_registered_operation_runs_through_torch_compile(self) -> None:
        torch.manual_seed(1234)
        scores = torch.randn(
            1,
            2,
            8,
            8,
            device="cuda",
            requires_grad=True,
        )
        keep_mask = torch.ones(
            1,
            1,
            8,
            8,
            device="cuda",
            dtype=torch.bool,
        ).tril()

        compiled_softpick = torch.compile(softpick, fullgraph=True)
        output = compiled_softpick(scores, keep_mask)
        output.square().sum().backward()

        self.assertEqual(output.shape, scores.shape)
        self.assertIsNotNone(scores.grad)
        self.assertTrue(bool(torch.isfinite(scores.grad).all()))


def print_test_environment() -> None:
    """Print the software and GPU identity needed to interpret the run."""
    try:
        triton_version = importlib.metadata.version("triton")
    except importlib.metadata.PackageNotFoundError:
        triton_version = "not installed"

    print(f"PyTorch version      : {torch.__version__}")
    print(f"CUDA build           : {torch.version.cuda}")
    print(f"Triton version       : {triton_version}")
    print(f"CUDA available       : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU name             : {torch.cuda.get_device_name(0)}")
        print(
            "Compute capability   : "
            f"{torch.cuda.get_device_capability(0)}"
        )


print_test_environment()
complete_suite = unittest.TestSuite()
complete_suite.addTests(
    unittest.defaultTestLoader.loadTestsFromTestCase(SoftpickReferenceTests)
)
complete_suite.addTests(
    unittest.defaultTestLoader.loadTestsFromTestCase(SoftpickTritonTests)
)
suite_result = unittest.TextTestRunner(verbosity=2).run(complete_suite)
if not suite_result.wasSuccessful():
    raise AssertionError("The Chapter 8 correctness suite failed")


## 6. Compare eager, compiled, and handwritten forward paths

This benchmark checks every candidate against the eager oracle before it
records any timing. It reports the first synchronized call separately from
warmed execution because compilation and kernel initialization belong to a
different question than steady-state throughput. Keep the shape, dtype,
epsilon, warm-up count, software versions, and GPU identity with every
result.

The benchmark is enabled automatically on a GPU runtime. Set
`RUN_SOFTPICK_BENCHMARK` to `False` when you want a faster notebook pass.


In [ ]:
SoftpickFunction = Callable[[torch.Tensor, torch.Tensor], torch.Tensor]


@dataclasses.dataclass
class SoftpickBenchmarkConfig:
    batch_size: int = 8
    heads: int = 4
    query_length: int = 256
    key_length: int = 256
    dtype: str = "float16"
    epsilon: float = DEFAULT_SOFTPICK_EPSILON
    warmup: int = 25
    steps: int = 100
    seed: int = 1234
    json_output: str | None = None


@dataclasses.dataclass
class Measurement:
    implementation: str
    cold_call_ms: float
    mean_forward_ms: float
    rows_per_second: float


def build_softpick_inputs(
    config: SoftpickBenchmarkConfig,
) -> tuple[torch.Tensor, torch.Tensor]:
    dtype = getattr(torch, config.dtype)
    generator = torch.Generator(device="cuda")
    generator.manual_seed(config.seed)
    scores = torch.randn(
        config.batch_size,
        config.heads,
        config.query_length,
        config.key_length,
        device="cuda",
        dtype=dtype,
        generator=generator,
    )
    query = torch.arange(config.query_length, device="cuda").view(-1, 1)
    key = torch.arange(config.key_length, device="cuda").view(1, -1)
    keep_mask = (key <= query).view(
        1, 1, config.query_length, config.key_length
    )
    validate_softpick_inputs(scores, keep_mask, config.epsilon)
    return scores, keep_mask


def synchronize_call(
    function: SoftpickFunction,
    scores: torch.Tensor,
    keep_mask: torch.Tensor,
) -> tuple[torch.Tensor, float]:
    torch.cuda.synchronize()
    started = time.perf_counter()
    output = function(scores, keep_mask)
    torch.cuda.synchronize()
    return output, (time.perf_counter() - started) * 1000.0


def measure_warmed_forward(
    name: str,
    function: SoftpickFunction,
    scores: torch.Tensor,
    keep_mask: torch.Tensor,
    config: SoftpickBenchmarkConfig,
    cold_call_ms: float,
) -> Measurement:
    for _ in range(config.warmup):
        function(scores, keep_mask)
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(config.steps):
        function(scores, keep_mask)
    end.record()
    torch.cuda.synchronize()

    mean_ms = start.elapsed_time(end) / config.steps
    rows = scores.numel() // scores.shape[-1]
    return Measurement(
        implementation=name,
        cold_call_ms=cold_call_ms,
        mean_forward_ms=mean_ms,
        rows_per_second=rows / (mean_ms / 1000.0),
    )


def run_softpick_benchmark(
    config: SoftpickBenchmarkConfig,
) -> dict[str, Any] | None:
    if not torch.cuda.is_available():
        print("SKIP: the Softpick benchmark requires a GPU runtime")
        return None
    if config.dtype == "bfloat16" and not torch.cuda.is_bf16_supported():
        raise RuntimeError("This GPU does not support bfloat16")
    if config.warmup < 1 or config.steps < 1:
        raise ValueError("warmup and steps must both be positive")
    if not math.isfinite(config.epsilon) or config.epsilon <= 0:
        raise ValueError("epsilon must be a finite positive number")

    output_path = Path(config.json_output) if config.json_output else None
    if output_path is not None and output_path.exists():
        raise FileExistsError(
            f"Refusing to overwrite an earlier benchmark: {output_path}"
        )

    scores, keep_mask = build_softpick_inputs(config)
    eager = lambda values, mask: softpick_reference(
        values, mask, config.epsilon
    )
    compiled = torch.compile(eager, fullgraph=True)
    triton_forward = lambda values, mask: softpick_triton_forward(
        values, mask, config.epsilon
    )
    candidates: list[tuple[str, SoftpickFunction]] = [
        ("eager", eager),
        ("compiled", compiled),
        ("triton", triton_forward),
    ]

    measurements: list[Measurement] = []
    with torch.inference_mode():
        expected, eager_cold_ms = synchronize_call(eager, scores, keep_mask)
        for name, function in candidates:
            if name == "eager":
                actual = expected
                cold_call_ms = eager_cold_ms
            else:
                actual, cold_call_ms = synchronize_call(
                    function, scores, keep_mask
                )
            reduced = scores.dtype in {torch.float16, torch.bfloat16}
            torch.testing.assert_close(
                actual,
                expected,
                rtol=2e-3 if reduced else 2e-4,
                atol=2e-3 if reduced else 2e-5,
            )
            measurements.append(
                measure_warmed_forward(
                    name,
                    function,
                    scores,
                    keep_mask,
                    config,
                    cold_call_ms,
                )
            )

    environment = {
        "pytorch": torch.__version__,
        "cuda_build": torch.version.cuda,
        "triton": importlib.metadata.version("triton"),
        "gpu": torch.cuda.get_device_name(0),
        "compute_capability": torch.cuda.get_device_capability(0),
        "shape": list(scores.shape),
        "dtype": str(scores.dtype),
        "epsilon": config.epsilon,
        "warmup": config.warmup,
        "steps": config.steps,
        "seed": config.seed,
    }
    result = {
        "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
        "environment": environment,
        "measurements": [dataclasses.asdict(row) for row in measurements],
    }

    print(json.dumps(environment, indent=2))
    print("implementation       cold call       warmed forward       rows/s")
    for row in measurements:
        print(
            f"{row.implementation:<18} "
            f"{row.cold_call_ms:>9.3f} ms "
            f"{row.mean_forward_ms:>14.4f} ms "
            f"{row.rows_per_second:>14,.0f}"
        )

    if output_path is not None:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with output_path.open("x", encoding="utf-8") as stream:
            json.dump(result, stream, indent=2)
            stream.write("\n")
        print(f"JSON result written to {output_path}")
    return result


In [ ]:
RUN_SOFTPICK_BENCHMARK = torch.cuda.is_available()
SOFTPICK_BENCHMARK = SoftpickBenchmarkConfig()

softpick_benchmark_result = (
    run_softpick_benchmark(SOFTPICK_BENCHMARK)
    if RUN_SOFTPICK_BENCHMARK
    else None
)


## 7. Exercise the operation in the TinyStories decoder

The remaining definitions provide byte-level tokenization, deterministic
token-window sampling, a small streamed TinyStories subset, mixed-precision
training, smoke tests, end-to-end benchmarks, and profiler export. The
defaults match the chapter workload: context 256, width 256, four heads,
four blocks, feed-forward width 1,024, batch size 8, and 200 training steps.

Dataset streaming takes a fixed prefix in stored order. For exact data
reproduction, set `dataset_revision` to a known Hugging Face commit. The
cache identity includes the dataset, revision, split, subset size, and
tokenizer version.


In [ ]:
# ---------------------------------------------------------------------------
# Tokenization and dataset preparation
# ---------------------------------------------------------------------------


def encode_story(text: str) -> list[int]:
    """Encode one story as UTF-8 bytes followed by the end-of-story token."""
    return list(text.encode("utf-8")) + [END_OF_STORY_TOKEN_ID]


def decode_tokens(token_ids: Iterable[int]) -> str:
    """Decode byte tokens, dropping the end-of-story marker."""
    payload = bytes(t for t in token_ids if t != END_OF_STORY_TOKEN_ID)
    return payload.decode("utf-8", errors="replace")


def describe_dataset(dataset: Any, revision: str | None) -> dict[str, Any]:
    """Collect whatever identifying metadata the dataset object exposes."""
    info = getattr(dataset, "info", None)
    description: dict[str, Any] = {
        "dataset_id": DATASET_ID,
        "requested_revision": revision,
        "builder_name": getattr(info, "builder_name", None),
        "config_name": getattr(info, "config_name", None),
        "version": str(getattr(info, "version", None)),
        "download_size": getattr(info, "download_size", None),
    }
    return description


def cache_key(
    split: str,
    num_stories: int,
    revision: str | None,
) -> str:
    """Stable cache identity. Contains no machine-specific paths."""
    identity = {
        "dataset_id": DATASET_ID,
        "revision": revision or "unpinned",
        "split": split,
        "num_stories": num_stories,
        "tokenizer_version": TOKENIZER_VERSION,
    }
    encoded = json.dumps(identity, sort_keys=True).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()[:16]


def build_token_stream(
    split: str,
    num_stories: int,
    revision: str | None,
    cache_dir: Path,
    verbose: bool = True,
) -> tuple[torch.Tensor, dict[str, Any]]:
    """Tokenize a fixed prefix of a TinyStories split into one token stream.

    Streaming mode reads records in stored file order, so taking the first
    `num_stories` records gives the same subset on every run without
    downloading the complete dataset.
    """
    cache_dir.mkdir(parents=True, exist_ok=True)
    key = cache_key(split, num_stories, revision)
    cache_path = cache_dir / f"tinystories_{split}_{key}.pt"

    if cache_path.exists():
        payload = torch.load(cache_path, map_location="cpu")
        if verbose:
            print(f"loaded cached {split} tokens from {cache_path.name}")
        return payload["tokens"], payload["metadata"]

    try:
        from datasets import load_dataset
    except ImportError as error:  # pragma: no cover - environment dependent
        raise SystemExit(
            "training on TinyStories requires the Hugging Face 'datasets' "
            "package. Install it with: pip install datasets"
        ) from error

    dataset = load_dataset(
        DATASET_ID,
        split=split,
        streaming=True,
        revision=revision,
    )

    token_ids: list[int] = []
    stories_read = 0
    for record in itertools.islice(dataset, num_stories):
        token_ids.extend(encode_story(record[DATASET_TEXT_FIELD]))
        stories_read += 1

    if stories_read < num_stories:
        print(
            f"warning: requested {num_stories} {split} stories but the split "
            f"yielded only {stories_read}"
        )

    tokens = torch.tensor(token_ids, dtype=torch.int64)
    metadata = describe_dataset(dataset, revision)
    metadata.update(
        {
            "split": split,
            "requested_stories": num_stories,
            "stories_read": stories_read,
            "num_tokens": int(tokens.numel()),
            "tokenizer_version": TOKENIZER_VERSION,
        }
    )
    torch.save({"tokens": tokens, "metadata": metadata}, cache_path)
    if verbose:
        print(f"cached {split} tokens to {cache_path.name}")
    return tokens, metadata


class TokenStreamBatcher:
    """Cut fixed-length input and shifted-target windows from a token stream."""

    def __init__(self, tokens: torch.Tensor, context_length: int) -> None:
        if tokens.numel() < context_length + 1:
            raise ValueError(
                f"token stream holds {tokens.numel()} tokens, which is fewer "
                f"than the {context_length + 1} needed for one window"
            )
        self.tokens = tokens
        self.context_length = context_length
        self.largest_offset = tokens.numel() - context_length - 1

    def batch_at(self, offsets: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        window = torch.arange(self.context_length)
        index = offsets.unsqueeze(1) + window.unsqueeze(0)
        return self.tokens[index], self.tokens[index + 1]

    def random_batch(
        self,
        batch_size: int,
        generator: torch.Generator,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        offsets = torch.randint(
            low=0,
            high=self.largest_offset + 1,
            size=(batch_size,),
            generator=generator,
        )
        return self.batch_at(offsets)

    def fixed_batches(
        self,
        batch_size: int,
        num_batches: int,
    ) -> list[tuple[torch.Tensor, torch.Tensor]]:
        """Evenly spaced windows, identical on every run."""
        total = batch_size * num_batches
        offsets = torch.linspace(0, self.largest_offset, total).round().long()
        return [
            self.batch_at(offsets[start : start + batch_size])
            for start in range(0, total, batch_size)
        ]


In [ ]:
@dataclasses.dataclass
class DecoderRunConfig:
    device: str = "auto"
    precision: str = "auto"
    compile: bool = False
    seed: int = 1234

    context_length: int = 256
    model_width: int = 256
    num_heads: int = 4
    num_layers: int = 4
    feedforward_width: int = 1024
    attention_dropout: float = 0.0
    residual_dropout: float = 0.0
    attention_normalizer: str = "softpick-reference"
    softpick_epsilon: float = DEFAULT_SOFTPICK_EPSILON

    train_stories: int = 2000
    validation_stories: int = 200
    dataset_revision: str | None = None
    cache_dir: str = ".cache/ch08"

    batch_size: int = 8
    train_steps: int = 200
    learning_rate: float = 3e-4
    weight_decay: float = 0.1
    gradient_clip: float = 1.0
    log_every: int = 20
    validate_every: int = 100
    validation_batches: int = 8
    checkpoint: str | None = None

    warmup_steps: int = 10
    benchmark_steps: int = 30
    profile_warmup: int = 3
    profile_active: int = 5
    profile_rows: int = 20
    profile_output: str | None = None


def validate_decoder_config(config: DecoderRunConfig) -> None:
    if config.model_width % config.num_heads != 0:
        raise ValueError("model_width must be divisible by num_heads")
    if config.attention_normalizer not in {
        "softmax", "softpick-reference", "softpick-triton"
    }:
        raise ValueError("unknown attention_normalizer")
    if not math.isfinite(config.softpick_epsilon) or config.softpick_epsilon <= 0:
        raise ValueError("softpick_epsilon must be a finite positive number")
    positive_fields = {
        "context_length": config.context_length,
        "batch_size": config.batch_size,
        "train_steps": config.train_steps,
        "warmup_steps": config.warmup_steps,
        "benchmark_steps": config.benchmark_steps,
    }
    for name, value in positive_fields.items():
        if value < 1:
            raise ValueError(f"{name} must be at least 1")
# Environment, seeding, precision
# ---------------------------------------------------------------------------


def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def select_device(requested: str) -> torch.device:
    if requested == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if requested == "cuda" and not torch.cuda.is_available():
        raise SystemExit("device='cuda' was requested but CUDA is unavailable")
    return torch.device(requested)


def select_precision(requested: str, device: torch.device) -> torch.dtype:
    """Resolve the autocast dtype, rejecting unsupported combinations."""
    if requested == "auto":
        return torch.float16 if device.type == "cuda" else torch.float32
    if requested == "float32":
        return torch.float32
    if device.type != "cuda":
        raise SystemExit(
            f"precision='{requested}' requires CUDA. Use float32 on "
            f"{device.type}."
        )
    if requested == "bfloat16":
        if not torch.cuda.is_bf16_supported():
            raise SystemExit(
                "precision='bfloat16' was requested but this GPU does not "
                "support bfloat16. Use float16 or float32."
            )
        return torch.bfloat16
    return torch.float16


def autocast_context(device: torch.device, dtype: torch.dtype):
    """Autocast on CUDA in reduced precision, otherwise plain float32."""
    if device.type != "cuda" or dtype is torch.float32:
        return contextlib.nullcontext()
    return torch.amp.autocast(device_type="cuda", dtype=dtype, enabled=True)


def make_gradient_scaler(device: torch.device, dtype: torch.dtype):
    enabled = device.type == "cuda" and dtype is torch.float16
    return torch.amp.GradScaler(device.type, enabled=enabled)


def maybe_compile(model: nn.Module, enable: bool) -> tuple[nn.Module, str]:
    if not enable:
        return model, "eager"
    if not hasattr(torch, "compile"):
        raise SystemExit(
            "compile=True was requested but this PyTorch build does not provide "
            "torch.compile. Upgrade PyTorch or drop the option."
        )
    try:
        compiled = torch.compile(model)
    except Exception as error:
        raise SystemExit(f"torch.compile failed on this system: {error}")
    return compiled, "compiled"


def print_run_summary(
    config: ModelConfig,
    model: ByteLevelDecoder,
    device: torch.device,
    dtype: torch.dtype,
    seed: int,
    execution_mode: str,
    dataset_metadata: dict[str, Any] | None,
) -> None:
    print("=" * 72)
    print(f"PyTorch version      : {torch.__version__}")
    print(f"CUDA available       : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU name             : {torch.cuda.get_device_name(0)}")
    print(f"Device               : {device}")
    print(f"Autocast dtype       : {dtype}")
    print(f"Random seed          : {seed}")
    print(f"Execution mode       : {execution_mode}")
    print(f"Model configuration  : {dataclasses.asdict(config)}")
    print(f"Trainable parameters : {model.count_parameters():,}")
    if dataset_metadata is not None:
        print(f"Dataset              : {json.dumps(dataset_metadata, default=str)}")
        if dataset_metadata.get("requested_revision") is None:
            print(
                "Dataset revision is unpinned. Exact reproduction of this run "
                "also requires setting dataset_revision to the commit "
                "hash recorded at the time of the run."
            )
    print("=" * 72)


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------


def evaluate(
    model: nn.Module,
    batches: list[tuple[torch.Tensor, torch.Tensor]],
    device: torch.device,
    dtype: torch.dtype,
) -> float:
    model.eval()
    total = 0.0
    with torch.no_grad():
        for inputs, targets in batches:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            with autocast_context(device, dtype):
                _, loss = model(inputs, targets)
            total += loss.detach().item()
    model.train()
    return total / max(len(batches), 1)


def train(args: DecoderRunConfig) -> None:
    device = select_device(args.device)
    dtype = select_precision(args.precision, device)
    seed_everything(args.seed)

    cache_dir = Path(args.cache_dir)
    train_tokens, train_metadata = build_token_stream(
        "train", args.train_stories, args.dataset_revision, cache_dir
    )
    validation_tokens, _ = build_token_stream(
        "validation", args.validation_stories, args.dataset_revision, cache_dir
    )

    config = build_config(args)
    model = ByteLevelDecoder(config).to(device)
    model, execution_mode = maybe_compile(model, args.compile)
    print_run_summary(
        config, model_of(model), device, dtype, args.seed, execution_mode,
        train_metadata,
    )

    train_batcher = TokenStreamBatcher(train_tokens, config.context_length)
    validation_batcher = TokenStreamBatcher(
        validation_tokens, config.context_length
    )
    validation_batches = validation_batcher.fixed_batches(
        args.batch_size, args.validation_batches
    )

    sampler = torch.Generator()
    sampler.manual_seed(args.seed)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=args.learning_rate,
        betas=(0.9, 0.95),
        weight_decay=args.weight_decay,
    )
    scaler = make_gradient_scaler(device, dtype)

    model.train()
    tokens_per_step = args.batch_size * config.context_length
    window_tokens = 0
    # Accumulate on the device so that logging, not every step, pays for the
    # host synchronization.
    window_loss = torch.zeros((), device=device)
    window_start = time.perf_counter()

    for step in range(1, args.train_steps + 1):
        inputs, targets = train_batcher.random_batch(args.batch_size, sampler)
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with autocast_context(device, dtype):
            _, loss = model(inputs, targets)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), args.gradient_clip)
        scaler.step(optimizer)
        scaler.update()

        window_loss += loss.detach()
        window_tokens += tokens_per_step

        if step % args.log_every == 0:
            if device.type == "cuda":
                torch.cuda.synchronize()
            elapsed = time.perf_counter() - window_start
            mean_loss = window_loss.item() / args.log_every
            print(
                f"step {step:>6}  train loss {mean_loss:.4f}"
                f"  tokens/s {window_tokens / max(elapsed, 1e-9):,.0f}"
            )
            window_loss = torch.zeros((), device=device)
            window_tokens = 0
            window_start = time.perf_counter()

        if args.validate_every > 0 and step % args.validate_every == 0:
            validation_loss = evaluate(model, validation_batches, device, dtype)
            print(f"step {step:>6}  validation loss {validation_loss:.4f}")
            window_start = time.perf_counter()

    final_validation = evaluate(model, validation_batches, device, dtype)
    print(f"final validation loss {final_validation:.4f}")
    print(
        "This is a short demonstration run. The resulting model is not "
        "trained to convergence and is not comparable with published "
        "language models."
    )

    if args.checkpoint:
        checkpoint_path = Path(args.checkpoint)
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(
            {
                "model_state": model_of(model).state_dict(),
                "config": dataclasses.asdict(config),
                "train_steps": args.train_steps,
                "seed": args.seed,
                "dataset": train_metadata,
            },
            checkpoint_path,
        )
        print(f"checkpoint written to {checkpoint_path}")


def model_of(model: nn.Module) -> ByteLevelDecoder:
    """Return the underlying module, unwrapping a compiled wrapper."""
    inner = getattr(model, "_orig_mod", model)
    assert isinstance(inner, ByteLevelDecoder)
    return inner


# ---------------------------------------------------------------------------
# Synthetic batches for smoke tests, benchmarks, and profiling
# ---------------------------------------------------------------------------


def synthetic_batch(
    batch_size: int,
    sequence_length: int,
    device: torch.device,
    generator: torch.Generator,
) -> tuple[torch.Tensor, torch.Tensor]:
    stream = torch.randint(
        low=0,
        high=TOKENIZER_VOCABULARY_SIZE,
        size=(batch_size, sequence_length + 1),
        generator=generator,
    )
    inputs = stream[:, :-1].contiguous().to(device)
    targets = stream[:, 1:].contiguous().to(device)
    return inputs, targets


# ---------------------------------------------------------------------------
# Smoke test
# ---------------------------------------------------------------------------


def smoke_test_on_device(
    device: torch.device,
    attention_normalizer: str,
    softpick_epsilon: float,
) -> None:
    print(f"-- smoke test on {device} with {attention_normalizer}")
    config = ModelConfig(
        context_length=32,
        model_width=32,
        num_heads=2,
        num_layers=2,
        feedforward_width=64,
        attention_normalizer=attention_normalizer,
        softpick_epsilon=softpick_epsilon,
    )
    model = ByteLevelDecoder(config).to(device)
    model.train()

    generator = torch.Generator()
    generator.manual_seed(0)
    batch_size = 2
    inputs, targets = synthetic_batch(
        batch_size, config.context_length, device, generator
    )

    logits, loss = model(inputs, targets)
    expected_shape = (batch_size, config.context_length, config.vocabulary_size)
    if tuple(logits.shape) != expected_shape:
        raise SystemExit(
            f"logit shape {tuple(logits.shape)} does not match the expected "
            f"{expected_shape}"
        )
    print(f"   logits {tuple(logits.shape)}")

    if loss is None or not torch.isfinite(loss):
        raise SystemExit(f"loss is not finite: {loss}")
    print(f"   loss {loss.detach().item():.4f}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    required = [
        "token_embedding.weight",
        "position_embedding.weight",
        "blocks.0.attention.query_projection.weight",
        "blocks.0.attention.output_projection.weight",
        "blocks.0.feedforward.input_projection.weight",
        "final_norm.weight",
    ]
    named = dict(model.named_parameters())
    for name in required:
        parameter = named.get(name)
        if parameter is None:
            raise SystemExit(f"expected parameter {name} is missing")
        if parameter.grad is None:
            raise SystemExit(f"parameter {name} received no gradient")
    print(f"   {len(required)} named parameters received gradients")

    checked = 0
    for name, parameter in named.items():
        if parameter.grad is None:
            continue
        if not torch.isfinite(parameter.grad).all():
            raise SystemExit(f"parameter {name} has a non-finite gradient")
        checked += 1
    print(f"   {checked} gradient tensors are finite")

    before = named["blocks.0.attention.query_projection.weight"].detach().clone()
    optimizer.step()
    after = named["blocks.0.attention.query_projection.weight"].detach()
    if torch.equal(before, after):
        raise SystemExit("the optimizer step did not change any weights")
    print("   one optimizer step completed")


def run_smoke_test(args: DecoderRunConfig) -> None:
    seed_everything(args.seed)
    device = select_device(args.device)
    print(f"PyTorch {torch.__version__}, CUDA available {torch.cuda.is_available()}")

    devices = [torch.device("cpu")]
    if device.type == "cuda":
        devices.append(device)
    for target in devices:
        smoke_test_on_device(
            target,
            args.attention_normalizer,
            args.softpick_epsilon,
        )

    print("PASS: all smoke-test checks succeeded")


# ---------------------------------------------------------------------------
# Benchmark
# ---------------------------------------------------------------------------


def build_config(args: DecoderRunConfig) -> ModelConfig:
    return ModelConfig(
        context_length=args.context_length,
        model_width=args.model_width,
        num_heads=args.num_heads,
        num_layers=args.num_layers,
        feedforward_width=args.feedforward_width,
        attention_dropout=args.attention_dropout,
        residual_dropout=args.residual_dropout,
        attention_normalizer=args.attention_normalizer,
        softpick_epsilon=args.softpick_epsilon,
    )


def training_step(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: Any,
    inputs: torch.Tensor,
    targets: torch.Tensor,
    device: torch.device,
    dtype: torch.dtype,
    gradient_clip: float,
) -> torch.Tensor:
    with autocast_context(device, dtype):
        _, loss = model(inputs, targets)
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), gradient_clip)
    scaler.step(optimizer)
    scaler.update()
    return loss


def run_benchmark(args: DecoderRunConfig) -> None:
    device = select_device(args.device)
    dtype = select_precision(args.precision, device)
    seed_everything(args.seed)

    config = build_config(args)
    model = ByteLevelDecoder(config).to(device)
    model, execution_mode = maybe_compile(model, args.compile)
    model.train()
    print_run_summary(
        config, model_of(model), device, dtype, args.seed, execution_mode, None
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate)
    scaler = make_gradient_scaler(device, dtype)
    generator = torch.Generator()
    generator.manual_seed(args.seed)
    inputs, targets = synthetic_batch(
        args.batch_size, config.context_length, device, generator
    )

    for _ in range(args.warmup_steps):
        training_step(
            model, optimizer, scaler, inputs, targets, device, dtype,
            args.gradient_clip,
        )
    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    if device.type == "cuda":
        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)
        start_event.record()
        for _ in range(args.benchmark_steps):
            training_step(
                model, optimizer, scaler, inputs, targets, device, dtype,
                args.gradient_clip,
            )
        end_event.record()
        torch.cuda.synchronize()
        total_seconds = start_event.elapsed_time(end_event) / 1000.0
    else:
        start = time.perf_counter()
        for _ in range(args.benchmark_steps):
            training_step(
                model, optimizer, scaler, inputs, targets, device, dtype,
                args.gradient_clip,
            )
        total_seconds = time.perf_counter() - start

    mean_step = total_seconds / args.benchmark_steps
    tokens_per_step = args.batch_size * config.context_length

    print("-- benchmark result")
    print(f"   device            {device}")
    if device.type == "cuda":
        print(f"   gpu               {torch.cuda.get_device_name(0)}")
    print(f"   execution mode    {execution_mode}")
    print(f"   autocast dtype    {dtype}")
    print(f"   batch size        {args.batch_size}")
    print(f"   sequence length   {config.context_length}")
    print(f"   model config      {dataclasses.asdict(config)}")
    print(f"   warmup steps      {args.warmup_steps}")
    print(f"   measured steps    {args.benchmark_steps}")
    print(f"   mean step time    {mean_step * 1000:.3f} ms")
    print(f"   tokens per second {tokens_per_step / mean_step:,.0f}")
    if device.type == "cuda":
        peak = torch.cuda.max_memory_allocated() / (1024 ** 2)
        print(f"   peak allocated    {peak:.1f} MiB")
    print(
        "   Compare only runs with matching software, hardware, model, "
        "normalizer, precision, and measurement settings."
    )


# ---------------------------------------------------------------------------
# Profiler
# ---------------------------------------------------------------------------


def run_profile(args: DecoderRunConfig) -> None:
    device = select_device(args.device)
    dtype = select_precision(args.precision, device)
    seed_everything(args.seed)

    config = build_config(args)
    model = ByteLevelDecoder(config).to(device)
    model, execution_mode = maybe_compile(model, args.compile)
    model.train()
    print_run_summary(
        config, model_of(model), device, dtype, args.seed, execution_mode, None
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate)
    scaler = make_gradient_scaler(device, dtype)
    generator = torch.Generator()
    generator.manual_seed(args.seed)
    inputs, targets = synthetic_batch(
        args.batch_size, config.context_length, device, generator
    )

    for _ in range(args.warmup_steps):
        training_step(
            model, optimizer, scaler, inputs, targets, device, dtype,
            args.gradient_clip,
        )
    if device.type == "cuda":
        torch.cuda.synchronize()

    activities = [torch.profiler.ProfilerActivity.CPU]
    if device.type == "cuda":
        activities.append(torch.profiler.ProfilerActivity.CUDA)

    schedule = torch.profiler.schedule(
        wait=1, warmup=args.profile_warmup, active=args.profile_active, repeat=1
    )
    total_iterations = 1 + args.profile_warmup + args.profile_active

    with torch.profiler.profile(
        activities=activities,
        schedule=schedule,
        record_shapes=True,
        profile_memory=True,
    ) as profiler:
        for _ in range(total_iterations):
            training_step(
                model, optimizer, scaler, inputs, targets, device, dtype,
                args.gradient_clip,
            )
            profiler.step()

    sort_key = "cuda_time_total" if device.type == "cuda" else "cpu_time_total"
    print(
        profiler.key_averages().table(
            sort_by=sort_key, row_limit=args.profile_rows
        )
    )

    if args.profile_output:
        output_dir = Path(args.profile_output)
        output_dir.mkdir(parents=True, exist_ok=True)
        trace_path = output_dir / "ch08_trace.json"
        profiler.export_chrome_trace(str(trace_path))
        print(f"chrome trace written to {trace_path}")


### Smoke-test both Softpick paths

Each smoke test checks the logit shape, finite loss, representative
gradients, all finite gradient tensors, and a successful optimizer update.
The registered path runs through the CPU fallback and, when available, the
CUDA Triton kernels.


In [ ]:
smoke_base = DecoderRunConfig()
for normalizer in ("softpick-reference", "softpick-triton"):
    smoke_config = dataclasses.replace(
        smoke_base,
        attention_normalizer=normalizer,
    )
    validate_decoder_config(smoke_config)
    run_smoke_test(smoke_config)


### Optional TinyStories training run

Training is off by default because it downloads data and performs 200
optimization steps. Set `RUN_TRAINING` to `True` after the smoke tests pass.
The selected normalizer is the registered Triton operation; change it to
`softpick-reference` for the readable PyTorch path. A short run
demonstrates integration only and does not train a publication-quality
language model.


In [ ]:
RUN_TRAINING = False
TRAINING_RUN = DecoderRunConfig(
    attention_normalizer="softpick-triton",
    # Set this to a TinyStories commit hash for exact data reproduction.
    dataset_revision=None,
    checkpoint=None,
)

if RUN_TRAINING:
    validate_decoder_config(TRAINING_RUN)
    train(TRAINING_RUN)
else:
    print("Training is disabled. Set RUN_TRAINING = True to run it.")


## 8. Optional end-to-end benchmark and profiler trace

The end-to-end benchmark includes the decoder forward pass, loss,
backward pass, gradient clipping, optimizer step, and gradient-scaler
update. Compare runs only when software, hardware, model shape, normalizer,
precision, compilation mode, warm-up, and measured-step count match.

The profiler uses scheduled CPU and CUDA activities and can export a Chrome
trace. Profiler durations are not benchmark results: tracing and replay add
overhead. Enable one optional action at a time after the correctness cells
pass.


In [ ]:
RUN_END_TO_END_BENCHMARK = False
RUN_PROFILER = False

END_TO_END_RUN = DecoderRunConfig(
    attention_normalizer="softpick-triton",
    compile=False,
    warmup_steps=10,
    benchmark_steps=30,
)
PROFILE_RUN = dataclasses.replace(
    END_TO_END_RUN,
    profile_output="output/ch08_softpick_profile",
)

if RUN_END_TO_END_BENCHMARK and RUN_PROFILER:
    raise ValueError("Enable the benchmark or profiler, not both at once")
if RUN_END_TO_END_BENCHMARK:
    validate_decoder_config(END_TO_END_RUN)
    run_benchmark(END_TO_END_RUN)
elif RUN_PROFILER:
    validate_decoder_config(PROFILE_RUN)
    run_profile(PROFILE_RUN)
else:
    print(
        "Optional end-to-end measurement is disabled. Enable one flag "
        "after the correctness checks pass."
    )


## What the completed path proves

A passing notebook establishes that the readable equation, standalone
kernels, registered operation, and decoder agree on the tested contracts.
It also shows that gradients cross the custom-operation boundary and reach
earlier attention parameters.

Timing still answers only the measured question. The microbenchmark
isolates row normalization, while the end-to-end benchmark includes the
full training step. Attribute a performance change to launches, memory
traffic, occupancy, or generated code only after inspecting a profiler or
compiler artifact from the same run.
